In [2]:
import cv2
import numpy as np
from ultralytics import YOLO

# Load your YOLO model
model = YOLO("model/runs/train/toy_animals_full/weights/best.pt")

# Grid configuration for virtual checkerboard
GRID_ROWS = 3
GRID_COLS = 2

def draw_virtual_grid(frame):
    """Draws a virtual 3x2 grid on the frame."""
    h, w = frame.shape[:2]
    cell_w = w // GRID_COLS
    cell_h = h // GRID_ROWS

    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            x1 = c * cell_w
            y1 = r * cell_h
            x2 = x1 + cell_w
            y2 = y1 + cell_h

            # Draw box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 255), 1)

            # Label cell coordinates
            text = f"({r}, {c})"
            cv2.putText(frame, text, (x1 + 10, y1 + 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)

def detect_animals(frame):
    """YOLO detect animals and return list: (label, cx, cy)."""
    results = model(frame)[0]
    detections = []

    for box in results.boxes:
        cls = int(box.cls[0])
        label = model.names[cls]

        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        detections.append((label, cx, cy))

    return detections

def get_grid_cell(cx, cy, frame_width, frame_height):
    """Return (row, col) of the grid cell where the point falls."""
    cell_w = frame_width // GRID_COLS
    cell_h = frame_height // GRID_ROWS

    col = cx // cell_w
    row = cy // cell_h

    # Clamp inside range
    row = min(max(row, 0), GRID_ROWS - 1)
    col = min(max(col, 0), GRID_COLS - 1)

    return int(row), int(col)

def main():
    cap = cv2.VideoCapture(0)

    print("Virtual checkerboard grid active.")
    print("Move your toy anywhere — YOLO will detect it and map it to a grid cell.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        display = frame.copy()
        h, w = frame.shape[:2]

        # Draw virtual checkerboard grid
        draw_virtual_grid(display)

        # Detect animals
        detections = detect_animals(frame)

        for label, cx, cy in detections:
            # Find grid cell
            row, col = get_grid_cell(cx, cy, w, h)

            # Draw detection point
            cv2.circle(display, (cx, cy), 7, (0, 255, 0), -1)

            # Label detection
            cv2.putText(display,
                        f"{label} in cell ({row}, {col})",
                        (cx - 50, cy - 20),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7, (0, 255, 255), 2)

        cv2.imshow("Virtual Checkerboard Vision System", display)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


Virtual checkerboard grid active.
Move your toy anywhere — YOLO will detect it and map it to a grid cell.

0: 480x640 (no detections), 415.2ms
Speed: 16.8ms preprocess, 415.2ms inference, 12.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 tiger, 1781.7ms
Speed: 5.0ms preprocess, 1781.7ms inference, 32.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 tiger, 885.5ms
Speed: 12.7ms preprocess, 885.5ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 tiger, 619.3ms
Speed: 4.1ms preprocess, 619.3ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 660.3ms
Speed: 3.0ms preprocess, 660.3ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 tiger, 635.7ms
Speed: 3.4ms preprocess, 635.7ms inference, 2.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 665.0ms
Speed: 3.0ms preprocess, 665.0ms inference, 1.3ms postprocess per image a